# 04 — Structured text ingestion

This lesson imports two **fictional** source excerpts into the Savoroute catalog. It uses the same source parser, extractor, validation, and catalog-admission functions as the CLI. The workflow runs offline.

**Objectives:** inspect the input contract, extract fields, verify literal evidence spans, reject unsupported claims, admit valid records, and query the resulting catalog. Model-based extraction uses the same validation path but requires a separately selected provider.

## 1. Load and validate source documents

A source document carries an external ID, item kind, source name, collection timestamp, and text. `source_url` is optional; no HTTP request occurs. Normalization converts line endings to LF and applies Unicode NFC. Source hashes are computed over normalized text.

In [ ]:
import json
from pathlib import Path

from savoroute.data.catalog import catalog_to_dict, load_catalog, parse_catalog
from savoroute.ingestion.extraction import LabeledTextExtractor, parse_extraction
from savoroute.ingestion.sources import IngestionError, parse_source_batch
from savoroute.retrieval.lexical import SearchFilters, search_catalog
from savoroute.services.ingestion import ingest_sources, write_ingestion

source_file = Path("data/sample/ingestion_sources.json")
batch = parse_source_batch(json.loads(source_file.read_text(encoding="utf-8")))
print(f"Dataset: {batch.dataset_name} (synthetic={batch.synthetic})")
for document in batch.documents:
    print(f"{document.kind:10} {document.source_id} sha256={document.sha256[:12]}…")
assert len(batch.documents) == 2

## 2. Extract the labeled fields

`LabeledTextExtractor` is deliberately limited to single-line `Field: value` records. This makes the offline example reproducible; it is **not** a free-text natural language parser. Field values and quotes use the same structure expected from a language-model extractor.

In [ ]:
extractor = LabeledTextExtractor()
record = extractor.extract(batch.documents[0])
print("Item ID:", record.item.item_id)
print("Title:", record.item.title)
print("Cuisine:", record.item.cuisines)
print("Ingredients:", record.item.ingredients)
print("Source:", record.source.source_name)
assert record.item.title == "Ginger Mushroom Rice"

## 3. Check the evidence offsets

Each value must appear inside a quote copied verbatim from the normalized source. Evidence offsets are half-open character spans: `source.text[start:end] == quote`. Quoted statements are not independently verified facts.

In [ ]:
for field, entries in record.evidence.items():
    for entry in entries:
        assert record.source.text[entry.start:entry.end] == entry.quote
        print(f"{field:14} {str(entry.value):22} {entry.start:3}:{entry.end:3} {entry.quote!r}")

## 4. Reject a claim not supported by source text

The validator is independent of extraction strategy. The following deliberately invalid candidate claims the recipe is gluten-free, but its quote supports only a vegetarian label. No partial record is admitted.

In [ ]:
candidate = {
    "title": {"value": "Ginger Mushroom Rice", "quote": "Title: Ginger Mushroom Rice"},
    "description": {
        "value": "Rice with ginger, mushrooms, and scallions.",
        "quote": "Description: Rice with ginger, mushrooms, and scallions."
    },
    "cuisines": [],
    "dietary_tags": [{"value": "gluten-free", "quote": "Dietary tags: vegetarian"}],
    "ingredients": [],
    "location": None,
    "price_tier": None,
}
try:
    parse_extraction(candidate, batch.documents[0], extractor_id="invalid-demonstration")
except IngestionError as exc:
    print(f"Rejected: {exc}")
else:
    raise AssertionError("Unsupported extracted claim was accepted")

## 5. Admit valid records into the existing catalog

Catalog admission checks every record and validates the complete merged catalog using the existing schema. Item IDs derive from source IDs and are stable across runs; duplicate IDs are rejected. The two inputs add one recipe and one restaurant to the twelve-item example.

In [ ]:
base_catalog = load_catalog()
result = ingest_sources(batch, extractor, base_catalog=base_catalog)
assert len(base_catalog.items) == 12
assert len(result.catalog.items) == 14
assert parse_catalog(catalog_to_dict(result.catalog)) == result.catalog
for new_record in result.records:
    print(new_record.item.item_id, new_record.item.kind, new_record.item.title)

## 6. Save the catalog and evidence together

The catalog retains the existing version-1 schema. Field attribution is stored in a separate provenance JSON document. Outputs are individually replaced atomically, but the two-file pair is not a database transaction. This notebook uses a temporary directory so reruns leave the repository clean.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as directory:
    output = Path(directory)
    write_ingestion(
        result,
        catalog_path=output / "catalog.json",
        provenance_path=output / "evidence.json",
    )
    reloaded = parse_catalog(json.loads((output / "catalog.json").read_text()))
    evidence = json.loads((output / "evidence.json").read_text())
    assert reloaded == result.catalog
    assert len(evidence["records"]) == 2
    print("Saved catalog and evidence:", len(reloaded.items), "items")

## 7. Search the ingested records

There is no separate notebook implementation of search. We use `search_catalog`, the same function already exposed through `DiscoveryService` and the CLI.

In [ ]:
hits = search_catalog(
    result.catalog.items,
    "ginger mushrooms",
    SearchFilters(kind="recipe"),
    method="bm25",
    limit=3,
)
for hit in hits:
    print(f"{hit.item.title:30} score={hit.score:.3f}")
assert hits[0].item.title == "Ginger Mushroom Rice"

## 8. Model-based extraction (optional)

For free-form text, `ModelTextExtractor` uses a `TextGenerator` implementation such as `OpenAITextGenerator` or `OllamaTextGenerator`. The model proposes a JSON record; the same validator checks every quoted field before admission. Live model calls require credentials or a running local model and are **not executed** in this notebook.

The CLI syntax, provider setup, input contract, and limitations are described in [`docs/ingestion.md`](../docs/ingestion.md). The next milestone will evaluate extraction quality on permitted, non-synthetic sources and address multiple sources referring to the same entity.